# CH2. SmartRedis Tutorial

## Preamble

In [1]:
import os
import numpy as np
import time
from pathlib import Path
import jax
import jax.numpy as jnp
import equinox as eqx
import onnx
import onnxruntime as ort
import jax2onnx
from onnx.defs import onnx_opset_version
from smartsim import Experiment
from smartredis import Client

In [2]:
# Remove inherited Slurm environment from the interactive parent job
for key in list(os.environ):
    if key.startswith(("SLURM_", "SBATCH_", "SRUN_")):
        os.environ.pop(key)

### Directories

In [3]:
PROJECT_DIR = Path("/scratch/project_2015384/Hanseul")
BASE_DIR = PROJECT_DIR / "Codes" / "SmartSim"
MAIN_DIR = BASE_DIR / "GettingStarted"
SCRIPT_DIR = MAIN_DIR / "scripts"

### Configuration

In [4]:
# System configuration
SEED = 42
key = jax.random.PRNGKey(SEED)
np.random.seed(SEED)
N_JOBS = len(list(os.sched_getaffinity(0)))

# SmartRedis configuration
REDIS_PORT = 2026

In [5]:
# ONNX configuration (SmartRedis supports ONNX opset version 20 / IR version 9)
OPSET_VERSION = 20
IR_VERSION = 9

### Dataset I/O

In [33]:
def get_files(model):
    """Return the output and error file paths of a SmartSim model."""
    model_path = Path(model.path)

    output_file = model_path / f"{model.name}.out"
    error_file = model_path / f"{model.name}.err"

    return output_file, error_file

### DNN Architecture

In [6]:
def _forward(layers, x, activation):
    for layer in layers[:-1]:
        x = activation(layer(x))
    return layers[-1](x)

In [7]:
def _build_layers(layer, key, in_dim, hidden_dims, out_dim, init="glorot"):
    dims = [in_dim] + hidden_dims + [out_dim]
    keys = jax.random.split(key, len(dims) - 1)
    return [
        layer(key=keys[i], in_features=dims[i], out_features=dims[i + 1], init=init)
        for i in range(len(dims) - 1)
    ]

In [8]:
class Linear(eqx.Module):
    weight: jax.Array
    bias: jax.Array

    def __init__(self, key, in_features, out_features, init="glorot"):
        if isinstance(init, str):
            init = {
                "glorot": jax.nn.initializers.glorot_normal(),
                "he": jax.nn.initializers.he_normal(),
                "lecun": jax.nn.initializers.lecun_normal(),
            }.get(init.lower(), jax.nn.initializers.glorot_normal())

        self.weight = init(key, (out_features, in_features))
        self.bias = jnp.zeros(out_features)

    def __call__(self, x):
        return self.weight @ x + self.bias

In [9]:
class DNN(eqx.Module):

    layers: list
    activation: callable = eqx.field(static=True)

    def __init__(self, key, input_dim, hidden_dims, output_dim,
                 activation=jax.nn.silu, init="glorot"):
        
        self.layers = _build_layers(
            layer=Linear, 
            key=key, 
            in_dim=input_dim, hidden_dims=hidden_dims, out_dim=output_dim, 
            init=init
        )

        self.activation = activation

    def __call__(self, x):
        return _forward(self.layers, x, self.activation)

    def predict(self, X):
        return jax.vmap(self)(X)

In [10]:
@eqx.filter_jit
def predict(model, X):
    return jax.vmap(model)(X)

In [11]:
def batch_forward(x):
    """Batched forward pass written with jax2onnx-supported primitives."""
    for layer in model.layers[:-1]:
        x = x @ layer.weight.T + layer.bias
        x = jnp.maximum(x, 0.0)

    final_layer = model.layers[-1]
    return x @ final_layer.weight.T + final_layer.bias

## Examples

### Tutorials For Orchestrator (SmartSim + SmartRedis)

In [12]:
# Orchestrator database
# Initialise experiment
exp = Experiment(
    name="tutorial-smartredis-local",
    launcher="local",
)

# Define orchestrator database
db = exp.create_database(
    db_nodes=1,
    port=REDIS_PORT,
    interface="lo", # lo, ib0, enp1s0f0, enp1s0f1
    inter_op_threads=1,
    intra_op_threads=N_JOBS // 2,
)

# Set up the experiment
exp.generate(db)

# Start the experiment
exp.start(db, block=True)

# Generate SmartRedis client to connect to the orchestrator database
client = Client(
    address=db.get_address()[0],  # Get the address of the orchestrator database
    cluster=False,  # Set to True if using a cluster of databases
)

SmartRedis Library@13-28-25:WARNING: Environment variable SR_LOG_FILE is not set. Defaulting to stdout
SmartRedis Library@13-28-25:WARNING: Environment variable SR_LOG_LEVEL is not set. Defaulting to INFO


In [13]:
# Generate a random input tensor
input_tensor = np.random.randint(0, 128, size=(3, 3, 5))
print("Input tensor:")
print(input_tensor)

# Store the input tensor in the orchestrator database
input_key = "input_tensor"
client.put_tensor(input_key, input_tensor)

# Retrieve the input tensor from the orchestrator database
retrieved_tensor = client.get_tensor(input_key)
print("\nRetrieved tensor:")
print(retrieved_tensor)

Input tensor:
[[[102  51  92  14 106]
  [ 71  60  20 102 121]
  [ 82  86  74  74  87]]

 [[116  99 103  23   2]
  [ 21  52   1  87 107]
  [ 29  37   1  63  59]]

 [[ 20  32  75  57  21]
  [124 107  88  48  90]
  [ 58 126  41 127  91]]]

Retrieved tensor:
[[[102  51  92  14 106]
  [ 71  60  20 102 121]
  [ 82  86  74  74  87]]

 [[116  99 103  23   2]
  [ 21  52   1  87 107]
  [ 29  37   1  63  59]]

 [[ 20  32  75  57  21]
  [124 107  88  48  90]
  [ 58 126  41 127  91]]]


#### Without Backend

In [14]:
# Define a simple model
input_dim = 4
hidden_dims = [16, 32, 16]
output_dim = 2
activation = jax.nn.silu

# Create a DNN model instance
model = DNN(
    key=key,
    input_dim=input_dim,
    hidden_dims=hidden_dims,
    output_dim=output_dim,
    activation=activation,
)

In [15]:
# Generate a random input tensor
raw_data = np.random.rand(int(1e6), input_dim).astype(np.float32)

# Store the input
input_data_key = "input_data"
client.put_tensor(input_data_key, raw_data)

# Retrieve the input tensor from the orchestrator database
retrieved_data = client.get_tensor(input_data_key)

# Predict using the model
predictions = predict(model, retrieved_data)
predictions = np.array(predictions)

# Store the predictions in the orchestrator database
predictions_key = "predictions"
client.put_tensor(predictions_key, predictions)

# Retrieve the predictions from the orchestrator database
retrieved_predictions = client.get_tensor(predictions_key)

print("\nPredictions:")
print(retrieved_predictions)

print("\nPrediction == Retrieved Predictions:", np.allclose(predictions, retrieved_predictions))


Predictions:
[[-0.03650146 -0.0511927 ]
 [-0.11945499  0.02024042]
 [-0.06497657 -0.01613318]
 ...
 [-0.1039623  -0.02069808]
 [-0.07419261  0.01936686]
 [-0.14256889  0.06092184]]

Prediction == Retrieved Predictions: True


In [16]:
# Check the round trip time
start_time = time.time()

# Workflow (store input -> retrieve input -> predict -> store predictions -> retrieve predictions)
# Store the input tensor in the orchestrator database
client.put_tensor(input_key, raw_data)

# Retrieve the input tensor from the orchestrator database
retrieved_input = client.get_tensor(input_key)

# Predict using the model
predictions = predict(model, retrieved_input)
predictions = np.array(predictions)

# Store the predictions in the orchestrator database
client.put_tensor(predictions_key, predictions)

# Retrieve the predictions from the orchestrator database
retrieved_predictions = client.get_tensor(predictions_key)

end_time = time.time()
round_trip_time = end_time - start_time

print(f"\nRound trip time (No-backend): {round_trip_time:.6f} seconds")
print("\nPrediction == Retrieved Predictions:", np.allclose(predictions, retrieved_predictions))


Round trip time (No-backend): 0.057406 seconds

Prediction == Retrieved Predictions: True


#### With Backend

In [17]:
# Export the model to ONNX format
model_name = "dnn_model"
onnx_model = jax2onnx.to_onnx(
    batch_forward,  # The function to convert to ONNX
    inputs=[("B", input_dim)],
    model_name=model_name,
    opset=OPSET_VERSION
)
onnx_model.ir_version = IR_VERSION  # SmartRedis requires ONNX IR version 9 for compatibility

# Check the ONNX model
onnx.checker.check_model(onnx_model)

# Save the ONNX model to a file
onnx_model_path = MAIN_DIR / f"{model_name}.onnx"
onnx.save(onnx_model, onnx_model_path)
print(f"\nONNX model saved to: {onnx_model_path}")

13:28:25 rc5183 root[3346364:MainThread] INFO Converting JAX function to ONNX model with parameters: model_name=dnn_model, opset=20, input_shapes=[('B', 4)], input_params=None
13:28:25 rc5183 root[3346364:MainThread] INFO Dynamic batch dimensions detected.

ONNX model saved to: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/dnn_model.onnx


In [18]:
# Define SmartRedis model
client.set_model_from_file(
    name=model_name,
    model_file=str(onnx_model_path),
    backend="ONNX",
    device="CPU",   
)

In [20]:
# Run the model using SmartRedis backend
start_time = time.time()

# Put the input tensor in the orchestrator database
client.put_tensor(input_key, raw_data)

# Run the model using SmartRedis backend
client.run_model(
    name=model_name,
    inputs=[input_key],
    outputs=[predictions_key]
)

# Retrieve the predictions from the orchestrator database
retrieved_predictions = client.get_tensor(predictions_key)

end_time = time.time()
round_trip_time = end_time - start_time

print("\nPredictions:")
print(retrieved_predictions)

print(f"\nRound trip time (SmartRedis backend): {round_trip_time:.6f} seconds")


Predictions:
[[-0.24157143  0.13281545]
 [-0.29919145  0.38559115]
 [-0.20985636  0.28469548]
 ...
 [-0.5538458   0.29861686]
 [-0.30442658  0.270742  ]
 [-0.43121928  0.48203427]]

Round trip time (SmartRedis backend): 0.073223 seconds


### Ensemble with SmartRedis

In [24]:
# Define new Redis port
REDIS_PORT = 2027

# Define paths for producer and consumer scripts
PRODUCER_PATH = str(SCRIPT_DIR / "getting_started_smartredis_ex2_producer.py")
CONSUMER_PATH = str(SCRIPT_DIR / "getting_started_smartredis_ex2_consumer.py")

In [26]:
# Initialise experiment
exp = Experiment(
    name="tutorial-smartredis-local-ensemble",
    launcher="local",
)

# Define orchestrator database
db = exp.create_database(
    db_nodes=1,
    port=REDIS_PORT,
    interface="lo", # lo, ib0, enp1s0f0, enp1s0f1
    inter_op_threads=1,
    intra_op_threads=N_JOBS // 2,
)

# Set up the experiment
exp.generate(db)

# Start the experiment
exp.start(db, block=True)

# Generate SmartRedis client to connect to the orchestrator database
client = Client(
    address=db.get_address()[0],  # Get the address of the orchestrator database
    cluster=False,  # Set to True if using a cluster of databases
)

In [29]:
# Define settings
settings_producer = exp.create_run_settings(
    exe="python3",
    exe_args=[PRODUCER_PATH, "--redis-port", str(REDIS_PORT)],
)

settings_consumer = exp.create_run_settings(
    exe="python3",
    exe_args=[CONSUMER_PATH, "--redis-port", str(REDIS_PORT)],
)

# Define the producer and consumer
producer = exp.create_ensemble(
    name="producer",
    replicas=2,
    run_settings=settings_producer,
)

consumer = exp.create_model(
    name="consumer",
    run_settings=settings_consumer,
)

In [30]:
# Register incoming entities
for i in range(len(producer.models)):
    consumer.register_incoming_entity(producer.models[i])

In [32]:
# Attach files to producer and consumer
producer.attach_generator_files(to_configure=[PRODUCER_PATH])
consumer.attach_generator_files(to_configure=[CONSUMER_PATH])

# Generate the experiment
exp.generate(producer, overwrite=True)
exp.generate(consumer, overwrite=True)

# Run the producer and consumer
exp.start(producer, consumer, block=True, summary=True)

13:41:13 rc5183 SmartSim[3346364:MainThread] INFO 

=== Launch Summary ===
Experiment: tutorial-smartredis-local-ensemble
Experiment Path: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/tutorial-smartredis-local-ensemble
Launcher: local
Models: 1
Database Status: active

=== Ensembles ===
producer
Members: 2
Batch Launch: None

=== Models ===
consumer
Executable: /ROIHU_TYKKY_isWZenb/miniforge/envs/env1/bin/python3
Executable Arguments: /scratch/project_2015384/Hanseul/Codes/SmartSim/GettingStarted/scripts/getting_started_smartredis_ex2_consumer.py --redis-port 2027





13:41:14 rc5183 SmartSim[3346364:JobManager] INFO producer_0(3607036): SmartSimStatus.STATUS_COMPLETED
13:41:14 rc5183 SmartSim[3346364:JobManager] INFO consumer(3607108): SmartSimStatus.STATUS_COMPLETED
13:41:16 rc5183 SmartSim[3346364:JobManager] INFO producer_1(3607072): SmartSimStatus.STATUS_COMPLETED


In [34]:
# Check outputs
outputfile, _ = get_files(consumer)

with open(outputfile, 'r') as fin:
    print(fin.read())

SmartRedis Library@13-41-13:WARNING: Environment variable SR_LOG_FILE is not set. Defaulting to stdout
SmartRedis Library@13-41-13:WARNING: Environment variable SR_LOG_LEVEL is not set. Defaulting to INFO
Tensor for producer_0 is: [[[[0.16825496 0.53504264 0.70907038]
   [0.44577444 0.99471537 0.84635119]
   [0.23728119 0.36716062 0.64022706]]]]
Tensor for producer_1 is: [[[[0.1884374  0.51548285 0.05687198]
   [0.53064782 0.7203031  0.0593595 ]
   [0.33131548 0.86891215 0.0021909 ]]]]



In [35]:
# Stop the experiment
exp.stop(db)